# 14.5 · Prophet 思想 / Prophet-Style Additive Models

> **课程定位 / Where this fits**
> 第 5 课，**Part 14 · 时间序列**。一种和 ARIMA 完全不同的预测哲学。
> Lesson 5, **Part 14 · Time Series**. A forecasting philosophy completely different from ARIMA.
>
> **Prophet**(Facebook 开源)的思路不是"用过去预测未来"(自回归), 而是把时间序列看成**几个可解释成分的叠加**:**趋势 + 季节 + 节假日 + 噪声**, 然后像**曲线拟合/回归**一样把它们拟合出来。好处: **可解释**(能看每个成分)、**好调**(参数有直观含义)、**对缺失值/异常/多重季节稳健**、不需要平稳化。它特别适合"有强季节和节假日效应的业务数据"。本课**从零实现 Prophet 的核心思想**——用**线性趋势 + Fourier 季节**做回归预测。
> **Prophet** (open-sourced by Facebook) doesn't "predict from the past" (autoregression); instead it sees a series as a **sum of interpretable components**: **trend + seasonality + holidays + noise**, fitting them like **curve-fitting/regression**. Benefits: **interpretable** (see each component), **easy to tune** (intuitive parameters), **robust to missing/outliers/multiple seasonalities**, no stationarization needed. Great for "business data with strong seasonal and holiday effects." We **implement Prophet's core idea from scratch** — regression with a **linear trend + Fourier seasonality**.
>
> 💼 **实战/面试视角**："Prophet 和 ARIMA 的本质区别 / Fourier 表示季节 / Prophet 什么时候好用 / 可加模型" 是预测岗常考。
> 💼 **Practical/interview angle:** "Prophet vs ARIMA / Fourier seasonality / when Prophet shines / additive models" — common.

> 📐 **符号约定 / Notation**
> - 可加模型 $y(t) = g(t) + s(t) + h(t) + \epsilon$ / additive: trend + season + holiday + noise
> - Fourier 季节 —— 用 sin/cos 之和表示周期模式 / seasonality as a sum of sines/cosines

> 💡 **面试相关 / Interview-relevant**
> - "Prophet 的可加模型结构(趋势+季节+节假日)"（出镜率 ★★★★）
> - "Prophet vs ARIMA 的本质区别(回归拟合 vs 自回归)"（★★★★★）
> - "Fourier 级数怎么表示季节性"（★★★★）
> - "Prophet 适合什么场景/优势"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解 Prophet 的可加模型哲学(成分叠加 + 回归拟合)。
   Understand Prophet's additive philosophy (sum of components + regression fit).
2. 掌握用 **Fourier 级数**表示季节性。
   Master representing seasonality with Fourier series.
3. **从零实现**线性趋势 + Fourier 季节的预测模型。
   Implement a linear-trend + Fourier-seasonality model from scratch.
4. 理解 Prophet 的优势与 vs ARIMA。
   Understand Prophet's advantages and vs ARIMA.

## 目录 / TOC
1. [Prophet 的可加哲学 ⭐](#1)
2. [Fourier 级数表示季节 ⭐](#2)
3. [从零实现:趋势+季节回归预测 ⭐](#3)
4. [Prophet vs ARIMA + 小结 ⭐](#4)


<a id="1"></a>
## 1. Prophet 的可加哲学 ⭐ / Prophet's Additive Philosophy

ARIMA 的思路是"**用序列自己的过去值和误差**预测"(自回归)。Prophet 完全不同——它把时间序列看成几个**可解释成分的叠加**, 直接作为**时间 $t$ 的函数**来拟合(本质是回归/曲线拟合)：
ARIMA "predicts from the series' own past values and errors" (autoregression). Prophet is different — it views the series as a **sum of interpretable components**, fit directly as a **function of time $t$** (essentially regression/curve-fitting):

$$y(t) = \underbrace{g(t)}_{\text{趋势}} + \underbrace{s(t)}_{\text{季节}} + \underbrace{h(t)}_{\text{节假日}} + \epsilon$$

- **趋势 $g(t)$**:长期走向, Prophet 用**分段线性**(允许在"变点 changepoint"处改变斜率)或逻辑增长(有饱和上限)。
  **Trend $g(t)$:** long-term direction; Prophet uses **piecewise linear** (slope can change at "changepoints") or logistic growth (with a cap).
- **季节 $s(t)$**:用 **Fourier 级数**(sin/cos 之和)表示周期模式, 可同时建模多种季节(年/周/日)。
  **Seasonality $s(t)$:** periodic patterns via a **Fourier series** (sum of sin/cos), handling multiple seasonalities (yearly/weekly/daily).
- **节假日 $h(t)$**:把已知的特殊日期(双十一、春节)作为额外回归项。
  **Holidays $h(t)$:** known special dates (Black Friday, holidays) as extra regressors.

**关键差异(面试)**:Prophet 是**把时间当特征做回归**(给我任意一个日期 $t$, 直接算出预测), 而非 ARIMA 那样依赖紧邻的历史值。所以它**天然能处理缺失值、异常值、不规则采样**, 而且每个成分都**可解释、可单独调**。代价: 不显式建模自相关(短期依赖), 纯靠成分拟合。
**Key difference (interview):** Prophet **treats time as a feature for regression** (give any date $t$, directly compute the forecast), unlike ARIMA's reliance on immediate past values. So it **naturally handles missing values, outliers, irregular sampling**, and each component is **interpretable and separately tunable**. Cost: it doesn't explicitly model autocorrelation (short-term dependence), relying purely on component fitting.


<a id="2"></a>
## 2. Fourier 级数表示季节 ⭐ / Fourier-Series Seasonality

怎么用"时间的函数"表示**周期性季节**? Prophet 的答案是 **Fourier 级数**:任何周期函数都能用**一组不同频率的 sin/cos 之和**近似。对周期为 $P$(年度月度数据 $P=12$)的季节:
How to express **periodic seasonality** as a function of time? Prophet's answer is a **Fourier series:** any periodic function can be approximated by a **sum of sines/cosines at different frequencies**. For period $P$ ($P=12$ for monthly-yearly):

$$s(t) = \sum_{k=1}^{K}\left[a_k \cos\!\frac{2\pi k t}{P} + b_k \sin\!\frac{2\pi k t}{P}\right]$$

- $k=1$ 是基频(一年一个周期), $k=2,3,\dots$ 是高频(一年内更细的波动)。**$K$(Fourier 阶数)越大→能表示越复杂的季节形状**(但太大易过拟合)。
  $k=1$ is the base frequency (one cycle/year), $k=2,3,\dots$ are higher frequencies (finer wiggles). **Larger $K$ → richer seasonal shapes** (too large overfits).
- 妙处: 这些 sin/cos 就是一组**固定的特征**, 系数 $a_k, b_k$ 用**普通线性回归**就能拟合——把"建模季节"变成了"线性回归"。
  The beauty: these sin/cos are **fixed features**; the coefficients $a_k, b_k$ are fit by **ordinary linear regression** — turning "model seasonality" into "linear regression."

下面可视化 Fourier 特征。
Let's visualize the Fourier features.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
{AP}

def fourier_features(t, period, K):
    """生成 K 阶 Fourier 特征(每阶一对 sin/cos) / K-order Fourier features."""
    feats = []
    for k in range(1, K+1):
        feats.append(np.sin(2*np.pi*k*t/period))         # 第k阶 sin / k-th sine
        feats.append(np.cos(2*np.pi*k*t/period))         # 第k阶 cos / k-th cosine
    return np.column_stack(feats)

t = np.arange(36)                                         # 3年(36个月) / 3 years
fig, ax = plt.subplots(figsize=(11, 3.6))
F = fourier_features(t, period=12, K=3)
for k in range(3):
    ax.plot(t, F[:, 2*k], label=f"sin (k={k+1})", alpha=0.8)
ax.axvline(12, color="gray", ls=":"); ax.axvline(24, color="gray", ls=":")
ax.set_xlabel("月"); ax.set_title("Fourier 季节特征: 不同频率的 sin/cos (k=1 一年一周期, k越大波动越细)")
ax.legend(); plt.tight_layout(); plt.show()
print("Fourier级数: 用一组不同频率的sin/cos之和近似任意周期模式; K阶=K对sin/cos")
print("关键: 这些是固定特征, 系数用普通线性回归拟合 → '建模季节'变成'线性回归'")


<a id="3"></a>
## 3. 从零实现:趋势+季节回归预测 ⭐ / Build From Scratch

把 Prophet 的核心拼起来(简化版): 设计矩阵 = **[截距, 线性趋势 $t$, Fourier 季节特征]**, 用**线性回归**拟合 log(序列)(乘法型数据取 log 变可加), 然后对未来的 $t$ 直接预测。这就是"把时间当特征做回归"。
Assemble Prophet's core (simplified): design matrix = **[intercept, linear trend $t$, Fourier seasonal features]**, fit log(series) (multiplicative → log makes it additive) by **linear regression**, then predict for future $t$. This is "treat time as a feature for regression."


In [ ]:
from sklearn.linear_model import LinearRegression
train, test = ts[:120], ts[120:]                          # 按时间切 / chronological split
n_train = len(train)

def design_matrix(t, K=4, period=12):
    """设计矩阵: [线性趋势, Fourier季节特征] / design matrix: [linear trend, Fourier features]."""
    trend = t.reshape(-1, 1)                              # 线性趋势项 / linear trend term
    seasonal = fourier_features(t, period, K)            # 季节特征 / seasonal features
    return np.hstack([trend, seasonal])

t_train = np.arange(n_train); t_test = np.arange(n_train, n_train+len(test))
X_train = design_matrix(t_train); X_test = design_matrix(t_test)
y_train = np.log(train.values)                            # log → 把乘法季节变可加 / log for additive
model = LinearRegression().fit(X_train, y_train)          # 线性回归拟合趋势+季节系数 / fit by regression
fc = np.exp(model.predict(X_test))                        # 预测未来并 exp 回原尺度 / forecast & exp back
mape = (np.abs((test.values - fc)/test.values)).mean()*100

fig, ax = plt.subplots(figsize=(11, 4))
train.plot(ax=ax, label="训练"); test.plot(ax=ax, label="真实", color="green")
ax.plot(test.index, fc, "r--", label="可加模型预测")
ax.legend(); ax.set_title(f"Prophet式可加模型(线性趋势+Fourier季节)预测: MAPE={mape:.1f}%")
plt.tight_layout(); plt.show()
print(f"预测 MAPE = {mape:.1f}%; 仅用'时间→趋势+Fourier季节'的线性回归就能预测(无需自回归)")


In [ ]:
# Prophet 最迷人之处: 可解释 — 把预测拆成各成分 / Prophet's charm: decompose the fit into components
t_all = np.arange(len(ts)); X_all = design_matrix(t_all)
coef = model.coef_; intercept = model.intercept_
trend_part = np.exp(intercept + coef[0]*t_all)                       # 仅趋势(指数化) / trend only
seasonal_log = X_all[:, 1:] @ coef[1:]                               # 仅季节(log尺度) / seasonal (log)
fig, axes = plt.subplots(2, 1, figsize=(11, 6))
axes[0].plot(ts.index, ts.values, alpha=0.4, label="原始")
axes[0].plot(ts.index, trend_part, "r", lw=2, label="学到的趋势 g(t)")
axes[0].legend(); axes[0].set_title("成分1: 趋势(平滑上升的基线)")
axes[1].plot(ts.index, seasonal_log); axes[1].axhline(0, color="gray", ls="--")
axes[1].set_title("成分2: 季节 s(t) (Fourier拟合的周期形状, log尺度的相对偏移)")
plt.tight_layout(); plt.show()
print("可加模型天生可解释: 能把预测拆成'趋势'+'季节'分别画出来(ARIMA很难这样直观分解)")
print("这正是Prophet受业务团队欢迎的原因: 每个成分都看得懂、可单独调(如手动设趋势变点、加节假日)")


<a id="4"></a>
## 4. Prophet vs ARIMA + 小结 ⭐ / Prophet vs ARIMA

**Prophet vs ARIMA(面试高频对比)**:
**Prophet vs ARIMA (high-frequency comparison):**

| | Prophet(可加回归) | ARIMA(自回归) |
|---|---|---|
| 思路 | 时间→趋势+季节+节假日 的回归拟合 | 用过去值和误差自回归 |
| 平稳要求 | 不需要 | 需要(差分) |
| 可解释 | **强**(成分可分解可调) | 较弱 |
| 多重季节 | **容易**(多组Fourier) | 难 |
| 缺失/异常 | **稳健** | 敏感 |
| 短期自相关 | 不显式建模(弱) | **显式建模(强)** |
| 调参 | 直观(参数有业务含义) | 需经验(定阶) |

**什么时候用 Prophet(实务)**:有**强季节性 + 节假日效应**的业务数据(网站流量、销量)、有**缺失/异常**、需要**可解释**、想**快速上手不调阶**。Prophet 把"让分析师也能做不错的预测"做得很好。
**When to use Prophet:** business data with **strong seasonality + holiday effects** (web traffic, sales), with **missing/outliers**, needing **interpretability**, and wanting a **quick start without order-tuning**. Prophet excels at "letting analysts forecast decently."

**真实 Prophet** 比我们的简化版多: 自动检测**趋势变点**、贝叶斯估计(给不确定性区间)、节假日效应、可调先验等。本课实现的是它的**核心可加思想**。
**Real Prophet** adds: automatic **changepoint** detection, Bayesian estimation (uncertainty intervals), holiday effects, tunable priors, etc. We implemented its **core additive idea**.

```
Prophet: 可加模型 y=趋势g(t)+季节s(t)+节假日h(t)+噪声; 把时间当特征做回归(非自回归)
趋势: 分段线性(可设变点)或逻辑增长; 季节: Fourier级数(K对sin/cos, 用线性回归拟合系数, 可多重季节)
关键差异: Prophet回归拟合(给任意t直接算预测) vs ARIMA依赖紧邻历史值
优势: 可解释(成分可分解可调)+稳健(缺失/异常)+多重季节易+好调(参数有业务含义)+无需平稳
劣势: 不显式建模短期自相关; 真实Prophet还有变点检测/贝叶斯区间/节假日
适用: 强季节+节假日的业务数据(流量/销量), 有缺失异常, 要可解释、快速起步
```

### 💡 面试速查 / Interview cheat-sheet
1. **可加模型**: y=趋势+季节+节假日+噪声; 时间当特征做回归。
   Additive: y=trend+season+holiday+noise; time as a regression feature.
2. **vs ARIMA**: Prophet回归拟合/可解释/稳健/多季节; ARIMA自回归/建模短期依赖/需平稳。
   vs ARIMA: Prophet regression/interpretable/robust/multi-season; ARIMA autoregressive/short-term/stationary.
3. **Fourier季节**: 用sin/cos之和表示周期, 系数用线性回归拟合; K控制复杂度。
   Fourier seasonality: sin/cos sum for periodicity, coefficients by regression; K controls complexity.
4. **优势**: 可解释+稳健(缺失异常)+多重季节+好调+无需平稳化。
   Pros: interpretable + robust + multi-seasonal + tunable + no stationarization.
5. **适用**: 强季节+节假日业务数据, 要快速可解释的预测。
   Use: strong-seasonal+holiday business data needing quick interpretable forecasts.

### 下一节 / Next
**14.6 LSTM/GRU 预测**——从统计方法转向**深度学习**。用 12.1 学过的 LSTM 做时序预测: 把序列切成**滑动窗口**(用过去 N 步预测下一步), 喂给 LSTM 学习复杂的非线性时序模式。我们会强调**缩放、滑窗、按时间划分防泄漏**等实战要点。
**14.6 LSTM/GRU Forecasting** — moving to **deep learning**. Use the LSTM (from 12.1) for forecasting: slice the series into **sliding windows** (predict the next step from the past N), feeding an LSTM to learn complex nonlinear patterns. We'll stress practical points: **scaling, windowing, chronological split to avoid leakage**.
